# Using the HuggingFace API
## NER training, testing and finetuning

useful resources:
* https://github.com/huggingface/notebooks/blob/main/examples/token_classification.ipynb

How to use this notebook:
* change the variables under 1. Imports and general settings > #main parameters
* run all cells 

## 0. Check that all requirements are fulfilled

* Is a GPU available? Can we use it?
  * use `nvidia-smi [-L]` (`free -m`, `htop`) for inspection
  * short notice to others (gitter, Standup)
* Are all necessary libraries installed and functioning?
  * in virtual environment (`pyenv activate <virtualenv>`) run `pip install transformers`, `pip install torch` (for PyTorch, TensorFlow would be an alternative), `pip install datasets` and install/upgrade further dependencies in case needed

## 1. Imports and general settings

In [1]:
#imports
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification
from transformers import pipeline
import torch
import json
from datetime import datetime

/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#set GPU else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


### Choose a base model
* [RoBERTa](https://huggingface.co/docs/transformers/model_doc/roberta)
* [ELECTRA](https://huggingface.co/docs/transformers/model_doc/electra)
* [BSB-BERT Modell](https://huggingface.co/dbmdz/bert-base-german-cased) `#german`
* Flair: https://huggingface.co/flair/ner-german bzw. https://huggingface.co/flair/ner-german-large `#ner` `#german`
* [hmBERT](https://huggingface.co/dbmdz/bert-base-historic-multilingual-cased) `#hist`
* [distilbert](https://huggingface.co/distilbert/distilbert-base-uncased)

In [3]:
#main parameters
task = "ner" # one of "ner", "pos" or "chunk" from token classification
model_checkpoint = "dbmdz/bert-base-historic-multilingual-cased" #e.g. "prajjwal1/bert-tiny" good for testing purposes
batch_size = 16
dataset_name = "stefan-it/HisGermaNER"
eval_strategy = "epoch"
learning_rate = 2e-5
num_train_epochs = 3
weight_decay = 0.01

In [4]:
#set path to model

now = datetime.now().strftime('%Y-%m-%d_%H-%M')
model_name = model_checkpoint.split("/")[-1]

model_path = f"{model_name}-finetuned-{task}-{now}"

In [5]:
#write all parameters to dict to save later
parameters_dict = dict(((k, eval(k)) for k in ("task","model_checkpoint", "batch_size", "dataset_name", "eval_strategy", "learning_rate", "num_train_epochs", "weight_decay")))
parameters_dict

{'task': 'ner',
 'model_checkpoint': 'dbmdz/bert-base-historic-multilingual-cased',
 'batch_size': 16,
 'dataset_name': 'stefan-it/HisGermaNER',
 'eval_strategy': 'epoch',
 'learning_rate': 2e-05,
 'num_train_epochs': 3,
 'weight_decay': 0.01}

## 2. Finetuning a Model
see also: [https://huggingface.co/docs/transformers/training](https://huggingface.co/docs/transformers/training) 

1. load and inspect a dataset from HF directly / load and inspect a dataset from another source
2. instantiate `Tokenizer` and tokenize dataset
3. load base model
4. set training hyperparameters
5. set evaluation metric
6. create Trainer object
7. fine-tune by calling `train()`

### 2.1a Load and inspect a dataset from HF directly

Datasets for Pretraining:
* OCR Volltext Basis aus der OCR Pipeline `#hist`
* [German DBMDZ BERT Corpus](https://huggingface.co/datasets/stefan-it/german-dbmdz-bert-corpus) `#german`
  
Datasets for NER Finetuning:
* sbb_ner_data (nach Konsistenzchecks) `#german` `#hist` `#ner`
* [HisGermaNER](https://huggingface.co/datasets/stefan-it/HisGermaNER) `#german` `#hist` `#ner`
* https://github.com/NEISSproject/NERDatasets `#german` `(#hist)` `#ner`
* [HIPE](https://github.com/hipe-eval/HIPE-2022-data) `#german` `#multilingual` `#hist` `#ner`
* https://github.com/UB-Mannheim/reichsanzeiger-nlp `#german` `#hist` `#ner`

In [6]:
# load a dataset
from datasets import load_dataset
import csv

dataset_train = load_dataset(dataset_name, delimiter="\t", quoting=csv.QUOTE_NONE, split='train')  
dataset_test = load_dataset(dataset_name, delimiter="\t", quoting=csv.QUOTE_NONE, split='test') 
dataset_val = load_dataset(dataset_name, delimiter="\t", quoting=csv.QUOTE_NONE, split='validation') 

### 2.1b Load data locally

In [ ]:
"""
#see also: https://huggingface.co/docs/datasets/package_reference/loading_methods
from datasets import load_dataset
import csv
ds = load_dataset('csv', data_files='splits_HisGermaNER_v0_test.tsv', delimiter="\t", quoting=csv.QUOTE_NONE)
"""

### 2.1c Inspect and preprocess dataset

In [36]:
# inspect dataset
print(dataset_train.features)

{'TOKEN': Value(dtype='string', id=None), 'NE-COARSE-LIT': Value(dtype='string', id=None), 'MISC': Value(dtype='string', id=None)}


In [8]:
type(dataset_train)

datasets.arrow_dataset.Dataset

In [9]:
label_list = set([x for x in dataset_train["NE-COARSE-LIT"] if x is not None])
label_list = [x for x in label_list]
label_list

['B-ORG', 'I-PER', 'B-PER', 'B-LOC', 'I-ORG', 'O', 'I-LOC']

In [21]:
# function to preprocess and clean all dataset splits in the same way
import pandas as pd
from datasets import Dataset
from datasets import Features, Sequence, Value, ClassLabel

def clean_dataset_split(dataset_split):
    #read as df
    df_dataset = pd.DataFrame(dataset_split)
    
    #identify sentence splits using EndOfSentence
    df_ends = df_dataset.loc[df_dataset["MISC"] == "EndOfSentence"]
    df_ends_idx = df_ends.index.tolist()

    #read into lists of tokens / tags per sentence while cleaning from structural information / comments
    sent_tokens = []
    sent_ner_tags = []
    
    for i, end_idx in enumerate(df_ends_idx):
        sent_start = df_ends_idx[i-1]
        sent_end = df_ends_idx[i]
        sent_tokens_check = df_dataset["TOKEN"][sent_start+1:sent_end+1].tolist()
        sent_ner_tags_check = df_dataset["NE-COARSE-LIT"][sent_start+1:sent_end+1].tolist()
        if sent_tokens_check != []:
            if "-DOCSTART-" in sent_tokens_check:
                docstart_indices = [i for i, x in enumerate(sent_tokens_check) if x == "-DOCSTART-"]
                comment_indices = [i for i, x in enumerate(sent_tokens_check) if x.startswith("# onb:")]
                indices_to_delete = docstart_indices + comment_indices
                sent_tokens_check = [i for j, i in enumerate(sent_tokens_check) if j not in indices_to_delete]
                sent_ner_tags_check = [i for j, i in enumerate(sent_ner_tags_check) if j not in indices_to_delete]
        sent_tokens.append(sent_tokens_check)
        sent_ner_tags.append(sent_ner_tags_check)
    
    sent_tokens = [x for x in sent_tokens if x != []]
    sent_ner_tags = [x for x in sent_ner_tags if x != []]

    if len(sent_tokens) == len(sent_ner_tags):
        pass
    else:
        print("the length of sent_tokens and sent_ner_tags lists are not the same - please check again!")

    #create new, minimal df based on lists
    df_dataset_updated = pd.DataFrame(
    {'id': range(len(sent_tokens)),
     'tokens': sent_tokens,
     'ner_tags': sent_ner_tags
    })

    #transform data into Datasets format and set labels for ClassLabel based on label_list
    dataset_updated = Dataset.from_pandas(df_dataset_updated)
    dataset_updated = dataset_updated.cast_column("ner_tags", Sequence(ClassLabel(names=label_list)))

    return dataset_updated

In [22]:
#clean all splits per function
dataset_train_cleaned = clean_dataset_split(dataset_train)
dataset_test_cleaned = clean_dataset_split(dataset_test)
dataset_val_cleaned = clean_dataset_split(dataset_val)

dataset_train_cleaned[0]

Casting the dataset: 100%|██████████████████████████████████████████████████| 392/392 [00:00<00:00, 24502.87 examples/s]


{'id': 0,
 'tokens': ['Jetzt',
  'scheint',
  'er',
  'eine',
  'großartige',
  'Idee',
  'niedergeschrieben',
  'zu',
  'haben',
  ',',
  'würdig',
  'dem',
  'Gehirne',
  'eines',
  'Helden',
  'entsprungen',
  'zi',
  'sein',
  ';',
  'denn',
  'sein',
  'Auge',
  'flammt',
  'begeistert',
  'auf',
  ',',
  'auf',
  'seine',
  'Wangen',
  'fliegt',
  'ein',
  'dunkles',
  'Roth',
  ',',
  'durch',
  'seinen',
  'ganzen',
  'Organismus',
  'fährt',
  'eine',
  'tiefe',
  'Erschütterung',
  '.'],
 'ner_tags': [5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5]}

### 2.2a Align labels with tokenized text (for NER/Sequence Labeling)

"we need to do some processing on our labels as the input ids returned by the tokenizer are longer than the lists of labels our dataset contain"

In [23]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

label_all_tokens = True

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"{task}_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [24]:
# tokenize all dataset splits
tokenized_dataset_train = dataset_train_cleaned.map(tokenize_and_align_labels, batched=True)
tokenized_dataset_test = dataset_test_cleaned.map(tokenize_and_align_labels, batched=True)
tokenized_dataset_val = dataset_val_cleaned.map(tokenize_and_align_labels, batched=True)

Map: 100%|███████████████████████████████████████████████████████████████████| 392/392 [00:00<00:00, 8888.01 examples/s]


### 2.3 load base model

In [25]:
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dbmdz/bert-base-historic-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 2.4 set training hyperparameters and evaluation metric

In [26]:
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [27]:
args = TrainingArguments(
    model_path, #TODO: adapt to sth like models_finetuned/...?
    eval_strategy = eval_strategy,
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_train_epochs,
    weight_decay=weight_decay,
)

In [28]:
# evaluation
metric = evaluate.load("seqeval") #load_metric has been removed, see https://discuss.huggingface.co/t/cant-import-load-metric-from-datasets/107524/2

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

"""
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)
"""

'\ndef compute_metrics(eval_pred):\n    logits, labels = eval_pred\n    predictions = np.argmax(logits, axis=-1)\n    return metric.compute(predictions=predictions, references=labels)\n'

In [29]:
# training

#training_args = TrainingArguments(output_dir="test_trainer", eval_strategy="epoch")

trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_dataset_train, 
    eval_dataset=tokenized_dataset_val, 
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

### 2.5 Finetuning

In [31]:
trainer.train()

#TODO: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
#TODO: add Parameter-Efficient Finetuning (`peft` library)

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.223914,0.467942,0.438819,0.452912,0.936497
2,No log,0.172609,0.596110,0.549578,0.571899,0.952315
3,No log,0.167047,0.635762,0.607595,0.621359,0.956718


/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=276, training_loss=0.16985256084497424, metrics={'train_runtime': 104.7446, 'train_samples_per_second': 41.873, 'train_steps_per_second': 2.635, 'total_flos': 392575294821600.0, 'train_loss': 0.16985256084497424, 'epoch': 3.0})

### 2.6 Evaluation

In [32]:
#overall eval metrics

overall_results = trainer.evaluate()

In [34]:
# eval metrics for each category

predictions, labels, _ = trainer.predict(tokenized_dataset_val)
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results_per_tag = metric.compute(predictions=true_predictions, references=true_labels)
results_per_tag

/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'LOC': {'precision': np.float64(0.6496674057649667),
  'recall': np.float64(0.6704805491990846),
  'f1': np.float64(0.6599099099099098),
  'number': np.int64(437)},
 'ORG': {'precision': np.float64(0.0),
  'recall': np.float64(0.0),
  'f1': np.float64(0.0),
  'number': np.int64(9)},
 'PER': {'precision': np.float64(0.621978021978022),
  'recall': np.float64(0.5637450199203188),
  'f1': np.float64(0.5914315569487985),
  'number': np.int64(502)},
 'overall_precision': np.float64(0.6357615894039735),
 'overall_recall': np.float64(0.6075949367088608),
 'overall_f1': np.float64(0.6213592233009709),
 'overall_accuracy': 0.9567182339648879}

In [35]:
#TODO: save parameter settings and evaluation results in one dict
metadata = {"parameters":parameters_dict, "overall_results":overall_results, "results_per_tag":results_per_tag}

with open(model_path + '/metadata.json', 'w') as fp:
    json.dump(metadata, fp, default=str)

In [ ]:
#trainer.save_model("path/to/model")